## Step 1: Initialize the PageIndex Client
Connect to the PageIndex API using your API key. This sets up the client that will handle document processing and question answering without using traditional vector embeddings.

In [ ]:
from pageindex import PageIndexClient
import os
pi_client = PageIndexClient(api_key="")
os.environ["OPENAI_API_KEY"] = ""


## Step 2: Define Benchmark Questions & LLM-Judge Evaluation Function
Create the answer key with 5 ground-truth questions and their expected answers from the budget PDF. Define an LLM-based judge function that scores AI responses on factual correctness (0-100), ensuring fair evaluation regardless of answer length or verbosity.

In [6]:
import os
from openai import OpenAI
import pandas as pd

# Ground truth benchmark questions
benchmark = [
    {
        "question": "Analyze the government's disinvestment performance over the last five years. What is the target for 2025-26, and how does the document characterize the success of achieving these targets in recent years?",
        "expected": "Disinvestment targets have not been achieved for five consecutive years. The target for 2025-26 is Rs 47,000 crore, which is lower than the 2024-25 target of Rs 50,000 crore. In 2024-25, the government is estimated to meet only 66% of its target."
    },
    {
        "question": "Summarize all major initiatives proposed for MSMEs and micro-enterprises in the 2025-26 budget, including credit guarantees, classification changes, and specific financial instruments.",
        "expected": "Key initiatives include: (i) Doubling investment and turnover limits for MSME classification, (ii) Increasing credit guarantee cover to Rs 10 crore for small enterprises and Rs 20 crore for startups and exporters, and (iii) Providing 10 lakh UPI-linked credit cards with a Rs 5 lakh limit for micro-enterprises registered on the Udyam portal."
    },
    {
        "question": "Explain the shift from customs duty to the Agriculture Infrastructure and Development Cess (AIDC) as mentioned in the Finance Bill. What is the stated impact of this shift on the revenue shared with states?",
        "expected": "While the overall tax on items like solar cells and motor vehicles remains similar, there is a shift from customs duty to AIDC cess. This results in a lower proportion of revenue being shared with states because cesses are not part of the shareable tax pool."
    },
    {
        "question": "Describe the new three-year pipeline requirement for infrastructure ministries and the specific maritime and aviation infrastructure missions announced. What are the targets for these missions?",
        "expected": "Infrastructure ministries must formulate a three-year pipeline of PPP projects. A Maritime Development Fund (Rs 25,000 crore corpus, 49% govt contribution) will be set up. A modified UDAN scheme aims to connect 120 new destinations and carry 4 crore passengers in the next 10 years."
    },
    {
        "question": "How does the budget justify the 18.5% increase in allocation for women and children's welfare? Mention the specific linkage to the Pradhan Mantri Awas Yojana and the ownership rules that drive this increase.",
        "expected": "The allocation is Rs 5,65,161 crore. The increase is justified by higher PMAY allocation, as the female head of the family must be the owner or co-owner of the house, thus qualifying the expenditure under women's welfare."
    }
]

def llm_judge(question, expected, predicted):
    """Use GPT as a fair judge to score answers on factual correctness (0-100)."""
    # Initialize client inside to ensure it picks up any updated environment variables
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        return 0.0
        
    client = OpenAI(api_key=api_key)
    
    prompt = (
        "You are an evaluation judge. Compare the predicted answer to the expected answer for the given question.\n"
        "\n"
        "Score the predicted answer from 0 to 100 based on FACTUAL CORRECTNESS only:\n"
        "- Does the predicted answer contain the key facts, numbers, and entities from the expected answer?\n"
        "- A detailed answer that includes the correct facts should score just as high as a concise one.\n"
        "- Only penalize if the predicted answer is factually wrong or missing the key information.\n"
        "\n"
        "Question: " + question + "\n"
        "Expected Answer: " + expected + "\n"
        "Predicted Answer: " + predicted + "\n"
        "\n"
        "Respond with ONLY a number between 0 and 100, nothing else."
    )

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        raw = response.choices[0].message.content.strip()
        # Extract only the numeric part if GPT adds extra text
        score = "".join(c for c in raw if c.isdigit() or c == ".")
        return float(score) if score else 0.0
    except Exception as e:
        print(f"  Judge Error: {e}")
        return 0.0


## Step 3: Submit Document for Processing
Upload the budget PDF to PageIndex for processing. The API returns a unique `doc_id` that we use to reference this document in future queries.

In [7]:
result = pi_client.submit_document("./budget.pdf")
primary_doc_id = result["doc_id"]


## Step 4: Check Document Processing Status
Verify that PageIndex has finished processing the uploaded document before we start asking questions.

In [8]:
status = pi_client.get_document(primary_doc_id)["status"]
if status == "completed":
    print('Document processing completed')
    

Document processing completed


## Step 5: Ask a Sample Question
Test the system with a general question to confirm it can retrieve and answer queries from the processed document.

In [9]:
response = pi_client.chat_completions(
    messages=[{"role": "user", "content": "What are the key findings in this document?"}],
    doc_id=primary_doc_id
)
 
print(response["choices"][0]["message"]["content"])

Here are the key findings from the **Union Budget 2025-26 Analysis**:

---

## 💰 Fiscal Overview
- **Total Expenditure**: Rs 50,65,345 crore — up **7.4%** from 2024-25 revised estimates.
- **Total Receipts** (excl. borrowings): Rs 34,96,409 crore — up **11.1%**.
- **Nominal GDP growth** projected at **10.1%**.
- **Fiscal Deficit**: Targeted at **4.4% of GDP** (down from 4.8% in 2024-25 RE).
- **Revenue Deficit**: Targeted at **1.5% of GDP** (down from 1.9%).
- **Outstanding Liabilities**: 56.1% of GDP; government aims to bring this to ~50% by March 2031.

---

## 🧾 Key Tax Proposals
- **Income Tax Revamp**: New tax regime slabs revised; income up to **Rs 12 lakh** gets a full 100% rebate (previously Rs 7 lakh).
- **TDS/TCS thresholds raised**: TDS on rent raised to Rs 6 lakh/year; TCS on remittances threshold increased to Rs 10 lakh.
- **Customs duty** reduced on select items but offset by a new Agriculture Infrastructure & Development Cess (AIDC).
- **Startup tax exemption** extended 

## Step 6: View Document Tree Structure
Retrieve and display the internal tree structure that PageIndex built from the document. This shows how PageIndex organizes the content without using vector embeddings.

In [10]:
tree_result = pi_client.get_tree(primary_doc_id)["result"]
print(tree_result)


[{'title': 'Union Budget 2025-26 Analysis', 'node_id': '0000', 'page_index': 1, 'text': '# Union Budget 2025-26 Analysis\n', 'nodes': [{'title': 'Budget Highlights', 'node_id': '0001', 'page_index': 1, 'text': '## Budget Highlights\n\n- **Expenditure**: The government is estimated to spend Rs 50,65,345 crore in 2025-26, 7.4% higher than the revised estimate of 2024-25. Interest payments account for 25% of the total expenditure, and 37% of revenue receipts.\n- **Receipts**: The receipts (other than borrowings) in 2025-26 are estimated to be Rs 34,96,409 crore, about 11.1% higher than the revised estimate of 2024-25. Tax revenue which forms major part of the receipts is also expected to increase by 11% over the revised estimate for 2024-25.\n- **GDP**: The government has estimated a nominal GDP growth rate of 10.1% in 2025-26 (i.e., real growth plus inflation).\n- **Deficits**: Revenue deficit in 2025-26 is targeted at 1.5% of GDP. This is lower than the revised estimate of 1.9% in 2024-

## Step 7: Run Benchmark Evaluation & Calculate Accuracy
Ask all 5 benchmark questions one by one. Each AI response is scored by an LLM judge on factual correctness (0-100), then the overall PageIndex RAG accuracy is calculated.

In [11]:
pageindex_results = []

# Question 1: What is the estimated nominal GDP growth rate for 2025-26?
item1 = benchmark[0]
response1 = pi_client.chat_completions(
    messages=[{"role": "user", "content": item1["question"]}],
    doc_id=primary_doc_id
)
answer1 = response1["choices"][0]["message"]["content"]
print(f"  Predicted answer1: { answer1[:100] }...")
score1 = llm_judge(item1["question"], item1["expected"], answer1)
pageindex_results.append({
    "Question": item1["question"],
    "Expected": item1["expected"],
    "Predicted": answer1,
    "Score": score1
})

# Question 2: What is the new rebate limit under the revised new income tax regime?
item2 = benchmark[1]
response2 = pi_client.chat_completions(
    messages=[{"role": "user", "content": item2["question"]}],
    doc_id=primary_doc_id
)
answer2 = response2["choices"][0]["message"]["content"]
print(f"  Predicted answer2: { answer2[:100] }...")
score2 = llm_judge(item2["question"], item2["expected"], answer2)
pageindex_results.append({
    "Question": item2["question"],
    "Expected": item2["expected"],
    "Predicted": answer2,
    "Score": score2
})

# Question 3: Which ministry received the highest budget allocation in 2025-26, and what is the amount?
item3 = benchmark[2]
response3 = pi_client.chat_completions(
    messages=[{"role": "user", "content": item3["question"]}],
    doc_id=primary_doc_id
)
answer3 = response3["choices"][0]["message"]["content"]
print(f"  Predicted answer3: { answer3[:100] }...")
score3 = llm_judge(item3["question"], item3["expected"], answer3)
pageindex_results.append({
    "Question": item3["question"],
    "Expected": item3["expected"],
    "Predicted": answer3,
    "Score": score3
})

# Question 4: What is the proposed fiscal deficit target as a percentage of GDP for 2025-26?
item4 = benchmark[3]
response4 = pi_client.chat_completions(
    messages=[{"role": "user", "content": item4["question"]}],
    doc_id=primary_doc_id
)
answer4 = response4["choices"][0]["message"]["content"]
print(f"  Predicted answer4: { answer4[:100] }...")
score4 = llm_judge(item4["question"], item4["expected"], answer4)
pageindex_results.append({
    "Question": item4["question"],
    "Expected": item4["expected"],
    "Predicted": answer4,
    "Score": score4
})

# Question 5: What is the total allocation for the Pradhan Mantri Awas Yojana (Rural + Urban) in the new budget?
item5 = benchmark[4]
response5 = pi_client.chat_completions(
    messages=[{"role": "user", "content": item5["question"]}],
    doc_id=primary_doc_id
)
answer5 = response5["choices"][0]["message"]["content"]
print(f"  Predicted answer5: { answer5[:100] }...")
score5 = llm_judge(item5["question"], item5["expected"], answer5)
pageindex_results.append({
    "Question": item5["question"],
    "Expected": item5["expected"],
    "Predicted": answer5,
    "Score": score5
})

pageindex_df = pd.DataFrame(pageindex_results)

pageindex_accuracy = pageindex_df["Score"].mean()


pageindex_df = pd.DataFrame(pageindex_results)
pageindex_accuracy = pageindex_df["Score"].mean()
print(f"Pageindex RAG Accuracy: {round(pageindex_accuracy, 2)} %")
pageindex_df.to_csv("pageindex_results.csv", index=False)
print(f"Results saved to pageindex_results.csv")
pageindex_df


  Predicted answer1: Here is a comprehensive analysis of the government's disinvestment performance based on the document...
  Predicted answer2: Here is a comprehensive summary of all major MSME and micro-enterprise initiatives in the Union Budg...
  Predicted answer3: Here is a clear breakdown of what the document says on this topic:

---

## The Customs Duty → AIDC ...
  Predicted answer4: Here's a detailed breakdown of the three areas you asked about from the Union Budget 2025-26:

---

...
  Predicted answer5: Here is a precise breakdown of how the budget justifies the **18.5% increase** in women and children...
Pageindex RAG Accuracy: 100.0 %
Results saved to pageindex_results.csv


,Question,Expected,Predicted,Score
0,Analyze the government's disinvestment perform...,Disinvestment targets have not been achieved f...,Here is a comprehensive analysis of the govern...,100.0
1,Summarize all major initiatives proposed for M...,Key initiatives include: (i) Doubling investme...,Here is a comprehensive summary of all major M...,100.0
2,Explain the shift from customs duty to the Agr...,While the overall tax on items like solar cell...,Here is a clear breakdown of what the document...,100.0
3,Describe the new three-year pipeline requireme...,Infrastructure ministries must formulate a thr...,Here's a detailed breakdown of the three areas...,100.0
4,How does the budget justify the 18.5% increase...,"The allocation is Rs 5,65,161 crore. The incre...",Here is a precise breakdown of how the budget ...,100.0
